In [1]:
!pip install SimpleITK numpy matplotlib tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.6/52.6 MB 11.6 MB/s eta 0:00:00


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
def read_mhd_header(mhd_path):
    header = {}
    with open(mhd_path, 'r') as f:
        for line in f:
            if '=' in line:
                # Split only on the first occurrence of '='
                key, value = line.strip().split('=', 1)
                key = key.strip()
                value = value.strip()
                header[key] = value
    return header

mhd_file = "/content/drive/MyDrive/drr_generation_files/sub-verse601_img.mhd"
header = read_mhd_header(mhd_file)

print("📌 Extracted .mhd Header Info:")
for k, v in header.items():
    print(f"{k}: {v}")

📌 Extracted .mhd Header Info:
ObjectType: Image
NDims: 3
BinaryData: True
BinaryDataByteOrderMSB: False
CompressedData: False
TransformMatrix: 1 0 0 0 -1 0 0 0 1
Offset: -206.05199999999999 333.89800000000002 -672.59000000000003
CenterOfRotation: 0 0 0
AnatomicalOrientation: RPI
ElementSpacing: 0.90234400000000003 0.90234400000000003 0.90000000000000002
ITK_FileNotes: Time=133341.000
ITK_InputFilterName: MetaImageIO
ITK_original_direction: 1 0 0 0 -1 0 0 0 1
ITK_original_spacing: 0.902344 0.9 0.9
aux_file: ThAbd_WT_0_9_i6
bitpix: 16
cal_max: 0
cal_min: 0
datatype: 4
descrip: Time=133341.000
dim[0]: 3
dim[1]: 512
dim[2]: 512
dim[3]: 759
dim[4]: 1
dim[5]: 1
dim[6]: 1
dim[7]: 1
intent_code: 0
intent_p1: 0
intent_p2: 0
intent_p3: 0
pixdim[0]: -1
pixdim[1]: 0.902344
pixdim[2]: 0.902344
pixdim[3]: 0.9
pixdim[4]: 0
pixdim[5]: 0
pixdim[6]: 0
pixdim[7]: 0
qform_code: 1
qoffset_x: 206.052
qoffset_y: -333.898
qoffset_z: -672.59
quatern_b: 0
quatern_c: 1
quatern_d: 0
scl_inter: -1024
scl_slope: 1


In [4]:
import SimpleITK as sitk
import numpy as np

img = sitk.ReadImage("/content/drive/MyDrive/drr_generation_files/sub-verse601_img.mhd")
vol = sitk.GetArrayFromImage(img).astype(np.float32)   # (z,y,x)
spacing = np.array(img.GetSpacing())                    # (sx,sy,sz)
origin = np.array(img.GetOrigin())                      # (ox,oy,oz)
direction = np.array(img.GetDirection()).reshape(3,3)   # 3×3 matrix
inv_dir = np.linalg.inv(direction)

print("Volume shape (z,y,x):", vol.shape)
print("Spacing:", spacing)
print("Origin:", origin)
print("Direction:\n", direction)

def voxel_index_to_world(i, j, k):
    idx = np.array([i, j, k], dtype=np.float32) * spacing
    return origin + direction @ idx

def world_to_voxel_index(x, y, z):
    world = np.array([x, y, z], dtype=np.float32)
    idx = inv_dir @ (world - origin) / spacing
    return idx  # not rounded, for interpolation


Volume shape (z,y,x): (759, 512, 512)
Spacing: [0.902344 0.902344 0.9     ]
Origin: [-206.052  333.898 -672.59 ]
Direction:
 [[ 1.  0.  0.]
 [ 0. -1.  0.]
 [ 0.  0.  1.]]


In [5]:
nz, ny, nx = vol.shape
chunk = 64

printed = False

for z0 in range(0, nz, chunk):
    z1 = min(z0 + chunk, nz)
    print(f"Processing slice chunk z = {z0} → {z1}")

    # Only debug first chunk with first 10 voxels
    if not printed:
        print("\nFirst 10 voxel index → world coord mappings:\n")
        count = 0
        for k in range(z0, z1):
            for j in range(ny):
                for i in range(nx):
                    idx = np.array([i, j, k], dtype=np.float32)
                    world = origin + direction @ (idx * spacing)
                    print(f"Voxel index {idx.astype(int)} -> World {world}")
                    count += 1
                    if count >= 10:
                        break
                if count >= 10:
                    break
            if count >= 10:
                break

        printed = True

print("✓ Debug world coordinate check complete — no giant array created")


Processing slice chunk z = 0 → 64

First 10 voxel index → world coord mappings:

Voxel index [0 0 0] -> World [-206.052  333.898 -672.59 ]
Voxel index [1 0 0] -> World [-205.149656  333.898    -672.59    ]
Voxel index [2 0 0] -> World [-204.247312  333.898    -672.59    ]
Voxel index [3 0 0] -> World [-203.344968  333.898    -672.59    ]
Voxel index [4 0 0] -> World [-202.442624  333.898    -672.59    ]
Voxel index [5 0 0] -> World [-201.54028  333.898   -672.59   ]
Voxel index [6 0 0] -> World [-200.637936  333.898    -672.59    ]
Voxel index [7 0 0] -> World [-199.735592  333.898    -672.59    ]
Voxel index [8 0 0] -> World [-198.833248  333.898    -672.59    ]
Voxel index [9 0 0] -> World [-197.930904  333.898    -672.59    ]
Processing slice chunk z = 64 → 128
Processing slice chunk z = 128 → 192
Processing slice chunk z = 192 → 256
Processing slice chunk z = 256 → 320
Processing slice chunk z = 320 → 384
Processing slice chunk z = 384 → 448
Processing slice chunk z = 448 → 512
Pro

In [6]:
print("Min HU:", np.min(vol))
print("Max HU:", np.max(vol))
print("Mean HU:", np.mean(vol))


Min HU: -10925.0
Max HU: 32767.0
Mean HU: -7376.119


In [9]:
print(vol)

[[[-10925. -10925. -10925. ... -10925. -10925. -10925.]
  [-10925. -10925. -10925. ... -10925. -10925. -10925.]
  [-10925. -10925. -10925. ... -10925. -10925. -10925.]
  ...
  [-10925. -10925. -10925. ... -10925. -10925. -10925.]
  [-10925. -10925. -10925. ... -10925. -10925. -10925.]
  [-10925. -10925. -10925. ... -10925. -10925. -10925.]]

 [[-10925. -10925. -10925. ... -10925. -10925. -10925.]
  [-10925. -10925. -10925. ... -10925. -10925. -10925.]
  [-10925. -10925. -10925. ... -10925. -10925. -10925.]
  ...
  [-10925. -10925. -10925. ... -10925. -10925. -10925.]
  [-10925. -10925. -10925. ... -10925. -10925. -10925.]
  [-10925. -10925. -10925. ... -10925. -10925. -10925.]]

 [[-10925. -10925. -10925. ... -10925. -10925. -10925.]
  [-10925. -10925. -10925. ... -10925. -10925. -10925.]
  [-10925. -10925. -10925. ... -10925. -10925. -10925.]
  ...
  [-10925. -10925. -10925. ... -10925. -10925. -10925.]
  [-10925. -10925. -10925. ... -10925. -10925. -10925.]
  [-10925. -10925. -10925.

In [17]:
# volume shape (z, y, x)
nz, ny, nx = vol.shape
dim = np.array([nx, ny, nz], dtype=float)

ct_center = origin + direction @ ((dim - 1) * spacing / 2.0)
print("CT Center:", ct_center)


CT Center: [  24.496892  103.349108 -331.49    ]


In [18]:
# volume shape (z, y, x)
nz, ny, nx = vol.shape
dim = np.array([nx/2, ny/2, nz/2], dtype=float)

ct_center = origin + direction @ ((dim - 1) * spacing)
print("CT Center:", ct_center)


CT Center: [  24.04572  103.80028 -331.94   ]


In [21]:
drr_width  = 512      # pixels
drr_height = 512      # pixels
pixel_size = 1     # mm per pixel

# world size of detector
detector_width_mm  = drr_width  * pixel_size
detector_height_mm = drr_height * pixel_size


In [26]:
camera_distance = -1000  # mm
camera_pos = ct_center + np.array([camera_distance,0,0])  # +X

detector_distance = camera_distance   # symmetric projection
detector_center = ct_center - np.array([detector_distance,0,0])  # -X


In [27]:
print("camera_distance ",camera_pos);
print("detector center ",detector_center);

camera_distance  [-975.95428  103.80028 -331.94   ]
detector center  [1024.04572  103.80028 -331.94   ]


In [28]:
import numpy as np

# Shape: (z, y, x)
nz, ny, nx = vol.shape

# voxel corners (i,j,k)
corners_idx = np.array([
    [0, 0, 0],
    [nx-1, 0, 0],
    [0, ny-1, 0],
    [0, 0, nz-1],
    [nx-1, ny-1, 0],
    [nx-1, 0, nz-1],
    [0, ny-1, nz-1],
    [nx-1, ny-1, nz-1]
], dtype=np.float32)

def voxel_to_world(idx):
    return origin + (idx * spacing) @ direction.T

corners_world = np.array([voxel_to_world(c) for c in corners_idx])

print("World coordinate bounding box corners:")
for idx, world in zip(corners_idx, corners_world):
    print(f"Voxel {idx} -> {world}")


World coordinate bounding box corners:
Voxel [0. 0. 0.] -> [-206.052  333.898 -672.59 ]
Voxel [511.   0.   0.] -> [ 255.045784  333.898    -672.59    ]
Voxel [  0. 511.   0.] -> [-206.052    -127.199784 -672.59    ]
Voxel [  0.   0. 758.] -> [-206.052  333.898    9.61 ]
Voxel [511. 511.   0.] -> [ 255.045784 -127.199784 -672.59    ]
Voxel [511.   0. 758.] -> [255.045784 333.898      9.61    ]
Voxel [  0. 511. 758.] -> [-206.052    -127.199784    9.61    ]
Voxel [511. 511. 758.] -> [ 255.045784 -127.199784    9.61    ]


In [ ]:
import numpy as np

def index_to_world(idx, origin, spacing, direction):
    """
    Convert voxel index (i,j,k) -> world coordinate (x,y,z)

    idx: [i, j, k] voxel index
    origin: ITK origin (ox, oy, oz)
    spacing: voxel spacing (sx, sy, sz)
    direction: 3x3 direction matrix
    """
    idx = np.asarray(idx, dtype=float)
    return origin + (idx * spacing) @ direction.T


def world_to_index(world, origin, spacing, direction):
    """
    Convert world coordinate (x,y,z) -> voxel index (i,j,k)
    (fractional index, useful for interpolation)
    """
    world = np.asarray(world, dtype=float)
    inv_dir = np.linalg.inv(direction)
    return (inv_dir @ (world - origin)) / spacing
